In [28]:
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, pipeline
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder

In [29]:
dataset = load_dataset("Tobi-Bueck/customer-support-tickets", split="train")


In [30]:
dataset

Dataset({
    features: ['subject', 'body', 'answer', 'type', 'queue', 'priority', 'language', 'version', 'tag_1', 'tag_2', 'tag_3', 'tag_4', 'tag_5', 'tag_6', 'tag_7', 'tag_8'],
    num_rows: 61765
})

In [31]:
df = dataset.to_pandas()
df = df.dropna(subset=['subject', 'body', 'queue'])
# Combine Subject and Body
df['text'] = df['subject'] + " " + df['body']

In [32]:
df.head(5)

,subject,body,answer,type,queue,priority,language,version,tag_1,tag_2,tag_3,tag_4,tag_5,tag_6,tag_7,tag_8,text
0,Wesentlicher Sicherheitsvorfall,"Sehr geehrtes Support-Team,\n\nich möchte eine...",Vielen Dank für die Meldung des kritischen Sic...,Incident,Technical Support,high,de,51.0,Security,Outage,Disruption,Data Breach,None,None,None,None,Wesentlicher Sicherheitsvorfall Sehr geehrtes ...
1,Account Disruption,"Dear Customer Support Team,\n\nI am writing to...","Thank you for reaching out, <name>. We are awa...",Incident,Technical Support,high,en,51.0,Account,Disruption,Outage,IT,Tech Support,None,None,None,"Account Disruption Dear Customer Support Team,..."
2,Query About Smart Home System Integration Feat...,"Dear Customer Support Team,\n\nI hope this mes...",Thank you for your inquiry. Our products suppo...,Request,Returns and Exchanges,medium,en,51.0,Product,Feature,Tech Support,None,None,None,None,None,Query About Smart Home System Integration Feat...
3,Inquiry Regarding Invoice Details,"Dear Customer Support Team,\n\nI hope this mes...",We appreciate you reaching out with your billi...,Request,Billing and Payments,low,en,51.0,Billing,Payment,Account,Documentation,Feedback,None,None,None,Inquiry Regarding Invoice Details Dear Custome...
4,Question About Marketing Agency Software Compa...,"Dear Support Team,\n\nI hope this message reac...",Thank you for your inquiry. Our product suppor...,Problem,Sales and Pre-Sales,medium,en,51.0,Product,Feature,Feedback,Tech Support,None,None,None,None,Question About Marketing Agency Software Compa...


In [33]:
#Using LabelEncoder
le = LabelEncoder()
df['labels'] = le.fit_transform(df['queue'])

In [34]:
from imblearn.under_sampling import RandomUnderSampler

In [35]:
# Balance the dataset before feeding it to the model
ros = RandomUnderSampler(random_state=42)

In [36]:
# We reshape 'text' into a 2D array because imblearn expects it
X_resampled, y_resampled = ros.fit_resample(df[['text']], df['labels'])

# Create a new, perfectly balanced dataframe
df_balanced = pd.DataFrame(X_resampled, columns=['text'])
df_balanced['labels'] = y_resampled

print(f"Original dataset size: {len(df)}")
print(f"Balanced dataset size: {len(df_balanced)}")

Original dataset size: 56464
Balanced dataset size: 11960


In [37]:
# Create dictionaries so the model knows what the numbers mean!
# Example: id2label will look like {0: 'Billing', 1: 'Technical Support'...}
id2label = {int(i): label for i, label in enumerate(le.classes_)}
label2id = {label: int(i) for i, label in enumerate(le.classes_)}
num_unique_labels = len(le.classes_)


In [38]:
id2label,label2id

({0: 'Arts & Entertainment/Movies',
  1: 'Arts & Entertainment/Music',
  2: 'Autos & Vehicles/Maintenance',
  3: 'Autos & Vehicles/Sales',
  4: 'Beauty & Fitness/Cosmetics',
  5: 'Beauty & Fitness/Fitness Training',
  6: 'Billing and Payments',
  7: 'Books & Literature/Fiction',
  8: 'Books & Literature/Non-Fiction',
  9: 'Business & Industrial/Manufacturing',
  10: 'Customer Service',
  11: 'Finance/Investments',
  12: 'Finance/Personal Finance',
  13: 'Food & Drink/Groceries',
  14: 'Food & Drink/Restaurants',
  15: 'Games',
  16: 'General Inquiry',
  17: 'Health/Medical Services',
  18: 'Health/Mental Health',
  19: 'Hobbies & Leisure/Collectibles',
  20: 'Hobbies & Leisure/Crafts',
  21: 'Home & Garden/Home Improvement',
  22: 'Home & Garden/Landscaping',
  23: 'Human Resources',
  24: 'IT & Technology/Hardware Support',
  25: 'IT & Technology/Network Infrastructure',
  26: 'IT & Technology/Security Operations',
  27: 'IT & Technology/Software Development',
  28: 'IT Support',
  29

In [39]:
# Convert back to Hugging Face format and split
hf_dataset = Dataset.from_pandas(df_balanced)
split_data = hf_dataset.train_test_split(test_size=0.2, seed=42)
train_data = split_data['train']
test_data = split_data['test']

In [46]:
# STEP 2: Tokenization (Text to Numbers)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_data(example):
    return tokenizer(example["text"], padding="max_length", truncation=True, max_length=256)

train_data = train_data.map(tokenize_data, batched=True)
test_data = test_data.map(tokenize_data, batched=True)

Map:   0%|          | 0/9568 [00:00<?, ? examples/s]

Map:   0%|          | 0/2392 [00:00<?, ? examples/s]

In [47]:
# STEP 3: Setup the Model and Metric
# We pass our dictionaries into the model here
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_unique_labels,
    id2label=id2label,  # Model uses this to output text instead of numbers
    label2id=label2id
)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [48]:
def compute_accuracy(eval_pred):
    predictions, true_labels = eval_pred
    predicted_classes = predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(true_labels, predicted_classes)}

In [49]:
# STEP 4: Train!
training_args = TrainingArguments(
    output_dir="./balanced_distilbert_model",
    num_train_epochs=5,
    per_device_train_batch_size=32,   # Keeping the efficient batch size
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    fp16=True,                        # Keep this True for Google Colab's T4 GPU!
    learning_rate=5e-5,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True       # Automatically keeps the best version from the 5 epochs
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    compute_metrics=compute_accuracy
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,2.662926,0.390050
2,2.904786,1.227258,0.693562
3,2.904786,0.828523,0.763796
4,0.996712,0.670063,0.803094
5,0.996712,0.633835,0.813127


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=1495, training_loss=1.490145986295464, metrics={'train_runtime': 475.3482, 'train_samples_per_second': 100.642, 'train_steps_per_second': 3.145, 'total_flos': 3171445567979520.0, 'train_loss': 1.490145986295464, 'epoch': 5.0})

In [51]:
import numpy as np
from sklearn.metrics import classification_report

# Get predictions on the test set
raw_pred, _, _ = trainer.predict(test_data)
y_pred = np.argmax(raw_pred, axis=1)
y_true = test_data['labels']

# Print a detailed report using your LabelEncoder classes
print(classification_report(y_true, y_pred, target_names=le.classes_))

                                        precision    recall  f1-score   support

           Arts & Entertainment/Movies       0.90      0.91      0.91        47
            Arts & Entertainment/Music       0.98      0.95      0.96        43
          Autos & Vehicles/Maintenance       0.87      0.98      0.92        46
                Autos & Vehicles/Sales       0.90      0.80      0.84        44
            Beauty & Fitness/Cosmetics       0.93      0.95      0.94        43
     Beauty & Fitness/Fitness Training       0.96      1.00      0.98        43
                  Billing and Payments       0.57      0.71      0.63        48
            Books & Literature/Fiction       0.98      0.93      0.95        45
        Books & Literature/Non-Fiction       0.91      0.91      0.91        56
   Business & Industrial/Manufacturing       0.97      0.93      0.95        42
                      Customer Service       0.10      0.12      0.11        34
                   Finance/Investments 

In [60]:
from transformers import pipeline

# We pass the objects 'model' and 'tokenizer' directly from your memory
classifier = pipeline(
    "text-classification",
    model=model,           # The model object sitting in your RAM
    tokenizer=tokenizer,   # The tokenizer object sitting in your RAM
    device=0               # Use the GPU
)

# Test ticket
test_text = "Subject: Refund not processed | Body: I want my money back."

# Get the top 5 guesses
results = classifier(test_text, top_k=5)

print(f"--- Predictions for: {test_text} ---")
for res in results:
    print(f"Category: {res['label']} | Confidence: {res['score']:.4f}")

--- Predictions for: Subject: Refund not processed | Body: I want my money back. ---
Category: Billing and Payments | Confidence: 0.7186
Category: Shopping/E-commerce | Confidence: 0.0801
Category: Human Resources | Confidence: 0.0212
Category: Sales and Pre-Sales | Confidence: 0.0211
Category: Customer Service | Confidence: 0.0146


In [63]:
user_input = "Subject: Payment failed | Body: My card was declined but the money was deducted."

#Get and print the result
result = classifier(user_input)[0]

print(f"\nResult: {result['label']}")
print(f"Confidence: {result['score']:.4f}")


Result: Billing and Payments
Confidence: 0.8655
